# Dask cluster

This notebook starts the Dask cluster that all the other notebooks use. The cluster runs in its own background process (`dask_cluster.py`, which you don't need to edit), so restarting or shutting down kernels, including this one, does not stop it.

1. Change the settings in the next cell if needed, then run the cells down to the dashboard link.
2. In any other notebook, connect with:

   ```python
   from dask_cluster import get_client
   client = get_client()
   ```

3. When you are done, run the last cell (or `client.shutdown()` from any notebook). This stops the cluster and cancels the PBS worker jobs. The cluster also stops if you stop your JupyterHub server.

Everything the cluster writes goes to `$SCRATCH/dask`: the scheduler file, the cluster log (`dask_cluster.log`: dashboard link, worker job script, any startup errors), the PBS output of the worker jobs (`dask_logs/`), and the workers' spill space.

## Settings

New settings take effect the next time the cluster starts. To change a running cluster, stop it (last cell) and start it again.

In [ ]:
PROJECT = "PXXXXXXXX"   # your PBS project code
QUEUE = "casper"
N_WORKERS = 50          # number of PBS jobs (one worker each)
CORES = 1               # cores per worker
MEMORY_GB = 100         # memory per worker, in GB
WALLTIME = "12:00:00"   # the cluster stops when the worker jobs hit this limit

In [ ]:
import json
import os
import subprocess
import sys
import time
import warnings
warnings.filterwarnings("ignore")

from dask.distributed import Client

DASK_DIR = os.path.expandvars("$SCRATCH/dask")  # the other notebooks look for the cluster here
SCHEDULER_FILE = os.path.join(DASK_DIR, "scheduler.json")
LOG_FILE = os.path.join(DASK_DIR, "dask_cluster.log")
os.makedirs(DASK_DIR, exist_ok=True)

## Start the cluster

If a cluster is already running, this connects to it instead of starting another.

In [ ]:
# Connect to a running cluster if there is one
client = None
if os.path.exists(SCHEDULER_FILE):
    try:
        client = Client(scheduler_file=SCHEDULER_FILE, timeout="5s")
        print("A cluster is already running; connected to it.")
    except OSError:
        os.remove(SCHEDULER_FILE)  # stale file from a dead cluster

if client is None:
    # Start one in the background. start_new_session=True keeps it alive
    # when this kernel restarts or shuts down.
    settings = dict(PROJECT=PROJECT, QUEUE=QUEUE, N_WORKERS=N_WORKERS, CORES=CORES,
                    MEMORY_GB=MEMORY_GB, WALLTIME=WALLTIME, DASK_DIR=DASK_DIR)
    subprocess.Popen([sys.executable, "dask_cluster.py", json.dumps(settings)],
                     stdout=open(LOG_FILE, "w"), stderr=subprocess.STDOUT,
                     stdin=subprocess.DEVNULL, start_new_session=True)

    # Wait (up to 2 minutes) for the scheduler file, then connect
    for _ in range(120):
        if os.path.exists(SCHEDULER_FILE) or "Traceback" in open(LOG_FILE).read():
            break
        time.sleep(1)
    if not os.path.exists(SCHEDULER_FILE):
        raise RuntimeError("The cluster did not start. Log:\n\n" + open(LOG_FILE).read())
    client = Client(scheduler_file=SCHEDULER_FILE, timeout="30s")
client

Dashboard: open the Dask tab on the left of JupyterLab and enter `proxy/####`, where `####` is the port in the link below.

In [ ]:
client.dashboard_link

In [ ]:
# Optional: the cluster log (dashboard link, worker job script, any startup errors)
print(open(LOG_FILE).read())

## Stop the cluster

Run this when you are done. It stops the cluster for every notebook and cancels the worker jobs. Check afterwards with `qstat -u $USER` that no `dask-worker` jobs are left.

In [ ]:
client.shutdown()